#  Мониторинг процесса обучения

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann 
* https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html
* https://docs.wandb.ai/quickstart
* https://docs.wandb.ai/guides/track/log/log-summary#docusaurus_skipToContent_fallback
* https://docs.wandb.ai/guides/track/log/log-models
* https://www.youtube.com/playlist?list=PLD80i8An1OEGajeVo15ohAQYF1Ttle0lk

## Задачи для совместного разбора

In [1]:
%pip install wandb
%pip install scikit-learn seaborn torchmetrics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1\. Рассмотрите возможности пакета `wandb` по отслеживанию числовых значений, визуализации изображений и таблиц.

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Решите задачу регрессии, используя для мониторинга процесса обучения `wandb`. 

Разделите набор данных на обучающее и тестовое множество. В процессе обучения отслеживайте динамику изменения значения функции потерь и метрики $R^2$ по эпохам. После завершения обучения рассчитайте значение метрик MSE, RMSE, MAE и MAPE и сохраните в виде summary данного запуска. 

Обучите не менее трех моделей (с разной архитектурой или гиперпараметрами), отследите все запуски при помощи `wandb` и вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` с графиками обучения. Для каждого запуска приложите также скриншот с описанием гиперпараметров модели и summary (страница overview).

- [ ] Проверено на семинаре

In [2]:
import torch 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = torch.linspace(0, 1, 100).view(-1, 1)
y = torch.sin(2 * torch.pi * X) + 0.1 * torch.rand(X.size()) 

In [3]:

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# нормализация
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

device = "cuda" if torch.cuda.is_available() else "cpu"

class Net(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.model(x)


def train_model(config):
    wandb.init(project="regression-demo", config=config)

    model = Net(config.input_dim, config.hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.MSELoss()

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_te = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    for epoch in range(config.epochs):
        model.train()
        optimizer.zero_grad()

        preds = model(X_tr)
        loss = criterion(preds, y_tr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_te).cpu().numpy()
            val_true = y_test

            r2 = r2_score(val_true, val_preds)

        wandb.log({
            "epoch": epoch,
            "loss": loss.item(),
            "r2": r2
        })

    # финальные метрики
    model.eval()
    with torch.no_grad():
        preds = model(X_te).cpu().numpy().flatten()

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100

    wandb.summary["MSE"] = mse
    wandb.summary["RMSE"] = rmse
    wandb.summary["MAE"] = mae
    wandb.summary["MAPE"] = mape

    wandb.finish()

In [5]:
configs = [
    {"input_dim": 10, "hidden_dim": 16, "lr": 0.01, "epochs": 50},
    {"input_dim": 10, "hidden_dim": 32, "lr": 0.001, "epochs": 50},
    {"input_dim": 10, "hidden_dim": 64, "lr": 0.0005, "epochs": 50},
]

for cfg in configs:
    train_model(cfg)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your cho

ServicePollForTokenError: Failed to read port info after 30.0 seconds.

<p class="task" id="2"></p>

2\. Решите задачу классификации, используя для мониторинга процесса обучения `wandb`. 

Разделите набор данных на обучающее и тестовое множество. В процессе обучения отслеживайте динамику изменения значения функции потерь и метрики `Accuracy` по эпохам. После завершения обучения рассчитайте значение метрик Accuracy, Precision, Recall и F1 и сохраните в виде summary данного запуска. 

Отследите все запуски при помощи `wandb` и вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` с графиками обучения. Для каждого запуска приложите также скриншот с описанием гиперпараметров модели и summary (страница overview).


- [ ] Проверено на семинаре

In [ ]:
import torch as th
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=1000, noise=0.05, random_state=42)
X = th.FloatTensor(X)
y = th.LongTensor(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# нормализация
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_np = y_test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def train_model(config):
    wandb.init(project="classification-demo", config=config)

    model = Net(config.input_dim, config.hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.BCELoss()

    for epoch in range(config.epochs):
        # train
        model.train()
        optimizer.zero_grad()

        preds = model(X_train)
        loss = criterion(preds, y_train)
        loss.backward()
        optimizer.step()

        # eval
        model.eval()
        with torch.no_grad():
            preds_test = model(X_test).numpy()
            preds_bin = (preds_test > 0.5).astype(int)

            acc = accuracy_score(y_test_np, preds_bin)

        wandb.log({
            "epoch": epoch,
            "loss": loss.item(),
            "accuracy": acc
        })

    # финальные метрики
    model.eval()
    with torch.no_grad():
        preds_test = model(X_test).numpy()
        preds_bin = (preds_test > 0.5).astype(int)

    acc = accuracy_score(y_test_np, preds_bin)
    prec = precision_score(y_test_np, preds_bin)
    rec = recall_score(y_test_np, preds_bin)
    f1 = f1_score(y_test_np, preds_bin)

    wandb.summary["Accuracy"] = acc
    wandb.summary["Precision"] = prec
    wandb.summary["Recall"] = rec
    wandb.summary["F1"] = f1

    wandb.finish()

In [ ]:
configs = [
    {"input_dim": 30, "hidden_dim": 16, "lr": 0.01, "epochs": 50},
    {"input_dim": 30, "hidden_dim": 32, "lr": 0.001, "epochs": 50},
    {"input_dim": 30, "hidden_dim": 64, "lr": 0.0005, "epochs": 50},
]

for cfg in configs:
    train_model(cfg)

<p class="task" id="3"></p>

3\. Повторите задачу 2, вычислив и визуализировав матрицу несоответствий (для обучающей и тестовой выборки) тремя способами при помощи `wandb`:
* используя `torchmetrics` и представив данные в виде объекта `wandb.Table`;
* используя готовую функцию `wandb.plot.confusion_matrix`;
* построив тепловую карту при помощи `seaborn` и представив данные в виде объекта `wandb.Image`.

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.


- [ ] Проверено на семинаре

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def train_model(config):
    wandb.init(project="classification-demo", config=config)

    model = Net(config.input_dim, config.hidden_dim)
    
    optimizer = optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.BCELoss()

    for epoch in range(config.epochs):
        # train
        model.train()
        optimizer.zero_grad()

        preds = model(X_train)
        loss = criterion(preds, y_train)
        loss.backward()
        optimizer.step()

        # eval
        model.eval()
        with torch.no_grad():
            preds_test = model(X_test).numpy()
            preds_bin = (preds_test > 0.5).astype(int)

            acc = accuracy_score(y_test_np, preds_bin)

        wandb.log({
            "epoch": epoch,
            "loss": loss.item(),
            "accuracy": acc
        })

    # финальные метрики
    model.eval()
    with torch.no_grad():
        preds_test = model(X_test).numpy()
        preds_bin = (preds_test > 0.5).astype(int)

    acc = accuracy_score(y_test_np, preds_bin)
    prec = precision_score(y_test_np, preds_bin)
    rec = recall_score(y_test_np, preds_bin)
    f1 = f1_score(y_test_np, preds_bin)

    wandb.summary["Accuracy"] = acc
    wandb.summary["Precision"] = prec
    wandb.summary["Recall"] = rec
    wandb.summary["F1"] = f1

    wandb.finish()
    model.eval()
    with torch.no_grad():
        train_preds = model(X_train).numpy()
        test_preds = model(X_test).numpy()

    train_bin = (train_preds > 0.5).astype(int)
    test_bin = (test_preds > 0.5).astype(int)

In [ ]:
from torchmetrics.classification import BinaryConfusionMatrix

metric = BinaryConfusionMatrix()

# train
cm_train = metric(
    torch.tensor(train_bin),
    torch.tensor(y_train.numpy())
).numpy()

# test
cm_test = metric(
    torch.tensor(test_bin),
    torch.tensor(y_test_np)
).numpy()

# превращаем в таблицу
table_train = wandb.Table(
    data=cm_train.tolist(),
    columns=["Pred 0", "Pred 1"]
)

table_test = wandb.Table(
    data=cm_test.tolist(),
    columns=["Pred 0", "Pred 1"]
)

wandb.log({
    "conf_matrix_table_train": table_train,
    "conf_matrix_table_test": table_test
})

In [ ]:
wandb.log({
    "conf_mat_train": wandb.plot.confusion_matrix(
        y_true=y_train.numpy().flatten(),
        preds=train_bin.flatten()
    ),
    "conf_mat_test": wandb.plot.confusion_matrix(
        y_true=y_test_np,
        preds=test_bin.flatten()
    )
})

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# считаем
cm_train = confusion_matrix(y_train.numpy(), train_bin)
cm_test = confusion_matrix(y_test_np, test_bin)

# функция отрисовки
def plot_cm(cm, title):
    fig, ax = plt.subplots()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    return fig

fig_train = plot_cm(cm_train, "Train Confusion Matrix")
fig_test = plot_cm(cm_test, "Test Confusion Matrix")

wandb.log({
    "cm_heatmap_train": wandb.Image(fig_train),
    "cm_heatmap_test": wandb.Image(fig_test)
})

<p class="task" id="4"></p>

4\. Повторите задачу 2, обучив две модели: линейную и нелинейную. Для каждой из моделей сделайте прогноз (по всей выборке) и визуализируйте облако точек в виде `wandb.Image` (раскрасьте точки в цвета, соответствующие прогнозам модели).

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.


- [ ] Проверено на семинаре

<p class="task" id="5"></p>

5\. Повторите задачу 2, реализовав логику ранней остановки. Для этого разделите данные на три части: обучающую, валидационную и тестовую. Остановите процесс обучения, если целевая метрика (F1) на валидации не увеличивалась в течении последних $k$ ($k$ - гиперпараметр метода) эпох. В момент остановки выведите сообщение с текущим номером эпохи. Сохраните номер эпохи, на которой процесс обучения был прерван, в виде summary данного запуска.

Помимо отслеживания метрик на обучающей и тестовой выборке, также отслеживайте метрики на валидационной выборке в процессе обучения. 

Постройте таблицу `wandb.Table`, в которой содержится информация о:
* признаках объекта;
* правильном ответе;
* прогнозе модели;
* принадлежности к обучающему, валидационному или тестовому множеству.

Визуализируйте данную таблицу при помощи `wandb`.

Вставьте в текстовую ячейку скриншоты, демонстрирующие интерфейс `wandb` со всеми нужными визуализациями.

- [ ] Проверено на семинаре
